In [1]:
from pathlib import Path
import re
from pprint import pprint
from sentence_transformers import CrossEncoder
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [2]:
# Questions 目录
QUESTIONS_DIR = Path("./Questions")

# ———————————————————— 参数：拆分用 ————————————————————
# 8 个固定 section
SECTION_TITLES = [
    "Problem Interface",
    "Formal Definitions",
    "Required Complexity",
    "Maintained State",
    "Invariants",
    "Per-Operation Update Rules",
    "Output Rule",
    "Edge Cases and Consistency Checks",
]

SECTION_VARS = {
    "Problem Interface": "ProblemInterface",
    "Formal Definitions": "FormalDefinitions",
    "Required Complexity": "RequiredComplexity",
    "Maintained State": "MaintainedState",
    "Invariants": "Invariants",
    "Per-Operation Update Rules": "PerOperationUpdateRules",
    "Output Rule": "OutputRule",
    "Edge Cases and Consistency Checks": "EdgeCasesAndConsistencyChecks",
}

FIRST_SECTION_MARK = "### [1] Problem Interface"

SECTION_PATTERN = re.compile(
    r"^###\s*\[(\d+)\]\s*(.+?)\s*$",
    re.MULTILINE
)

NUMBERED_STEP_RE = re.compile(r"^\s*\d+\.\s+")



# ———————————————————— 参数：CrossEncoder计算用 ————————————————————
model = CrossEncoder("cross-encoder/stsb-roberta-large")

# 块名顺序
KEY_ORDER = [
    "ProblemInterface",                  # 1
    "FormalDefinitions",                 # 2
    "RequiredComplexity",                # 3
    "MaintainedState",                   # 4
    "Invariants",                        # 5
    "PerOperationUpdateRules",           # 6
    "OutputRule",                        # 7
    "EdgeCasesAndConsistencyChecks",     # 8
    "Note",                              # 9
]

# 对应矩阵变量名
MAT_NAMES = [
    "mat1PI",
    "mat2FD",
    "mat3RC",
    "mat4MS",
    "mat5INV",
    "mat6POUR",
    "mat7OR",
    "mat8ECCC",
    "mat9N",
]

# ———————————————————— 参数：数据分析 ————————————————————

TARGET_Q_INDEX = 0   # 指定打印 / 分析第几个题目（从 0 开始）

# k和temp必须严格与QtoE中的一致
k = 20
temperature_list = [0.0, 0.2, 0.5, 0.8]

ROUND_DIGITS = 6    # 保留几位

# 若只想看部分矩阵，可改成例如 ["mat8ECCC", "mat9N"]
SELECTED_MATS = None

DEFAULT_MAT_NAMES = [
    "mat1PI",    # Problem Interface
    "mat2FD",    # Formal Definitions
    "mat3RC",    # Required Complexity
    "mat4MS",    # Maintained State
    "mat5INV",   # Invariants
    "mat6POUR",  # Per-Operation Update Rules
    "mat7OR",    # Output Rule
    "mat8ECCC",  # Edge Cases and Consistency Checks
    "mat9N",
]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# 拆分用 函数部分
def list_question_dirs(questions_dir: Path):
    return sorted(
        [p for p in questions_dir.iterdir() if p.is_dir() and not p.name.startswith(".")],
        key=lambda x: x.name
    )


def list_explain_txts(explain_dir: Path):
    txts = []

    for p in explain_dir.iterdir():
        if not (p.is_file() and p.suffix.lower() == ".txt"):
            continue

        parts = p.stem.split()

        if len(parts) >= 1 and parts[0].isdigit():
            txts.append((int(parts[0]), p))

    txts_sorted = sorted(txts, key=lambda x: x[0])
    # display(txts_sorted)

    return [p for _, p in txts_sorted]


def remove_blank_lines(text: str) -> str:
    """
    对块内容做空行剔除：删除所有空行，仅保留非空行并按原顺序拼接。
    """
    if not text:
        return ""
    lines = [line for line in text.splitlines() if line.strip() != ""]
    return "\n".join(lines).strip()


def split_into_8_sections(text: str):
    """
    先切出 8 个块。
    舍弃 ### [1] Problem Interface 之前的任何内容。
    返回 dict:
    {
        'ProblemInterface': ...,
        ...
        'EdgeCasesAndConsistencyChecks': ...
    }
    """
    first_idx = text.find(FIRST_SECTION_MARK)
    if first_idx == -1:
        raise ValueError(f"未找到 '{FIRST_SECTION_MARK}'")

    text = text[first_idx:]
    matches = list(SECTION_PATTERN.finditer(text))
    if not matches:
        raise ValueError("未识别到 section 标题")

    parsed = {v: "" for v in SECTION_VARS.values()}

    for i, m in enumerate(matches):
        title = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        content = text[start:end]

        if title in SECTION_VARS:
            parsed[SECTION_VARS[title]] = content

    return parsed


def extract_note_from_section8(detailed_text: str):
    """
    从第八块中剥离第九块 Note。

    规则：
    1. 先对第八块原始内容处理，不提前删空行。
    2. 只考虑“最后一个由空行分隔出的连续非空块”是否应作为 Note。
    3. 若该最后块内部存在某行匹配 '数字. 空格'（如 5. xxx），
       则说明它属于第八块，不剥离。
    4. 若最后块内部没有编号行，且它前面由空行分隔，则将其剥离为 Note。
    5. 因为是按空行分块，若前面有编号步骤但中间已有空行断开，则最后块不再属于该步骤。
    """
    if not detailed_text.strip():
        return "", ""

    lines = detailed_text.splitlines()

    # 去掉末尾空行，便于找最后非空块
    end = len(lines) - 1
    while end >= 0 and lines[end].strip() == "":
        end -= 1

    if end < 0:
        return "", ""

    # 找最后一个连续非空块 [block_start, end]
    block_start = end
    while block_start >= 0 and lines[block_start].strip() != "":
        block_start -= 1
    block_start += 1

    # 如果整个 detailed 都是一个连续非空块，则不剥离 Note
    if block_start == 0:
        main_text = "\n".join(lines[:end + 1])
        return main_text, ""

    last_block_lines = lines[block_start:end + 1]

    # 若最后块内部含有编号步骤起始行，则它属于第八块
    if any(NUMBERED_STEP_RE.match(line) for line in last_block_lines):
        main_text = "\n".join(lines[:end + 1])
        return main_text, ""

    # 否则，最后块剥离为 Note
    note_text = "\n".join(last_block_lines)
    main_text = "\n".join(lines[:block_start - 1])  # block_start-1 是分隔空行

    return main_text, note_text


def parse_one_explain_file(txt_path: Path):
    """
    处理单个 explain 文件：
    1. 先切分出 8 个块
    2. 对第 8 块做第 9 块 Note 的剥离
    3. 对这 9 块内容统一做空行剔除
    返回：
    {
        'ProblemInterface': ...,
        ...
        'EdgeCasesAndConsistencyChecks': ...,
        'Note': ...
    }
    """
    raw = txt_path.read_text(encoding="utf-8")

    parsed8 = split_into_8_sections(raw)

    detailed_main, note_text = extract_note_from_section8(parsed8["EdgeCasesAndConsistencyChecks"])
    parsed8["EdgeCasesAndConsistencyChecks"] = detailed_main

    parsed9 = dict(parsed8)
    parsed9["Note"] = note_text

    # 最后统一对 9 块做空行剔除
    for key in parsed9:
        parsed9[key] = remove_blank_lines(parsed9[key])

    return parsed9


def parse_question_dir(question_dir: Path):
    """
    对单个题目目录处理。
    返回：
    {
        'ProblemSummary': [file1内容, file2内容, ...],
        ...
        'DetailedAlgorithmSteps': [...],
        'Note': [...]
    }
    """
    explain_dir = question_dir / "LLM Explains"
    if not explain_dir.exists() or not explain_dir.is_dir():
        raise FileNotFoundError(f"{question_dir} 下不存在 'LLM Explains' 文件夹")

    txt_files = list_explain_txts(explain_dir)

    result = {v: [] for v in SECTION_VARS.values()}
    result["Note"] = []

    for txt_path in txt_files:
        parsed = parse_one_explain_file(txt_path)
        for key in result:
            result[key].append(parsed[key])

    return result

In [4]:
# 遍历所有题目
dirs = list_question_dirs(QUESTIONS_DIR)
all_results = {}

for d in dirs:
    try:
        all_results[d.name] = parse_question_dir(d)
        print(f"done: {d.name}")
    except Exception as e:
        print(f"error: {d.name} -> {e}")

if all_results:
    first_name = next(iter(all_results))
    print(f"\n示例题目: {first_name}")
    pprint(all_results[first_name])

done: Easy B3666
done: Easy P15288
done: Easy P15457
done: Easy P4306
done: Easy P7714
done: Hard P11658
done: Hard P11823
done: Hard P13901
done: Hard P15082
done: Hard P6845
done: ML Q1
done: ML Q2
done: ML Q3
done: Mid P1407
done: Mid P14989
done: Mid P3007
done: Mid P3167
done: Mid P4092

示例题目: Easy B3666
{'EdgeCasesAndConsistencyChecks': ['- **Empty stack handling**: When `stack` '
                                   'is empty before step 2 (only at `k=1`), '
                                   'the while loop is skipped, `k=1` is '
                                   'pushed, and `ans = 0 XOR 1 = 1`, matching '
                                   'sample output first line.\n'
                                   '- **Equality case**: The condition '
                                   '`a[stack.top()] ≤ x[k]` uses non-strict '
                                   'inequality (`≤`), so equal values cause '
                                   'the old index to be popped. This is '
          

In [4]:
# 单独测试
dirs = list_question_dirs(QUESTIONS_DIR)
all_results = {}

d = dirs[0]
try:
    all_results[d.name] = parse_question_dir(d)
    print(f"done: {d.name}")
except Exception as e:
    print(f"error: {d.name} -> {e}")

if all_results:
    first_name = next(iter(all_results))
    print(f"\n示例题目: {first_name}")
    pprint(all_results[first_name])

[(1, PosixPath('Questions/Easy B3666/LLM Explains/1 0.0.txt')),
 (2, PosixPath('Questions/Easy B3666/LLM Explains/2 0.0.txt')),
 (3, PosixPath('Questions/Easy B3666/LLM Explains/3 0.0.txt')),
 (4, PosixPath('Questions/Easy B3666/LLM Explains/4 0.0.txt')),
 (5, PosixPath('Questions/Easy B3666/LLM Explains/5 0.0.txt')),
 (6, PosixPath('Questions/Easy B3666/LLM Explains/6 0.0.txt')),
 (7, PosixPath('Questions/Easy B3666/LLM Explains/7 0.0.txt')),
 (8, PosixPath('Questions/Easy B3666/LLM Explains/8 0.0.txt')),
 (9, PosixPath('Questions/Easy B3666/LLM Explains/9 0.0.txt')),
 (10, PosixPath('Questions/Easy B3666/LLM Explains/10 0.0.txt')),
 (11, PosixPath('Questions/Easy B3666/LLM Explains/11 0.0.txt')),
 (12, PosixPath('Questions/Easy B3666/LLM Explains/12 0.0.txt')),
 (13, PosixPath('Questions/Easy B3666/LLM Explains/13 0.0.txt')),
 (14, PosixPath('Questions/Easy B3666/LLM Explains/14 0.0.txt')),
 (15, PosixPath('Questions/Easy B3666/LLM Explains/15 0.0.txt')),
 (16, PosixPath('Questions/E

done: Easy B3666

示例题目: Easy B3666
{'EdgeCasesAndConsistencyChecks': ['- **Empty stack handling**: When `stack` '
                                   'is empty before step 2 (only at `k=1`), '
                                   'the while loop is skipped, `k=1` is '
                                   'pushed, and `ans = 0 XOR 1 = 1`, matching '
                                   'sample output first line.\n'
                                   '- **Equality case**: The condition '
                                   '`a[stack.top()] ≤ x[k]` uses non-strict '
                                   'inequality (`≤`), so equal values cause '
                                   'the old index to be popped. This is '
                                   'correct because the definition requires '
                                   '`a_i > a_j` for all `j > i`; if `a_i = '
                                   'a_j`, then `i` is not a suffix maximum.\n'
                                   '- **Single eleme

In [5]:
display(len(all_results))
display(len(all_results["Easy B3666"]))
display(len(all_results["Easy B3666"]["EdgeCasesAndConsistencyChecks"]))
display(all_results["Easy B3666"]["EdgeCasesAndConsistencyChecks"][0])
display(type(all_results))

18

9

80

'- **Empty stack handling**: When `stack` is empty before step 2 (only at `k=1`), the while loop is skipped, `k=1` is pushed, and `ans = 0 XOR 1 = 1`, matching sample output first line.\n- **Equality case**: The condition `a[stack.top()] ≤ x[k]` uses non-strict inequality (`≤`), so equal values cause the old index to be popped. This is correct because the definition requires `a_i > a_j` for all `j > i`; if `a_i = a_j`, then `i` is not a suffix maximum.\n- **Single element**: For `k=1`, `S_1 = {1}`, so `ans = 1`.\n- **Strictly decreasing input**: If `x[1] > x[2] > ... > x[n]`, then `stack = [1, 2, ..., k]` after `k` operations, and `ans = 1 XOR 2 XOR ... XOR k`.\n- **Strictly increasing input**: If `x[1] < x[2] < ... < x[n]`, then after `k` operations, `stack = [k]`, so `ans = k`.\n- **Parsing-unit consistency**: The input must be parsed as one fixed-length block of `n` integers on the second line; interpreting it as `n` separate scalar reads per line (e.g., one integer per line) would 

dict

In [6]:
def build_similarity_matrix(
    text_list,
    batch_size=64,
    show_pair_progress=False,
    matrix_label="",
    matrix_pos=None,
    matrix_total=None
):
    n = len(text_list)
    mat = np.zeros((n, n), dtype=float)

    # 对角线 = 1
    np.fill_diagonal(mat, 1.0)

    # 上三角 pair
    pairs = []
    index_pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((text_list[i], text_list[j]))
            index_pairs.append((i, j))

    total_pairs = len(pairs)

    if pairs:
        all_scores = []

        pair_pbar = None
        if show_pair_progress:
            desc = matrix_label if matrix_label else "pair progress"
            if matrix_pos is not None and matrix_total is not None:
                desc = f"[matrix {matrix_pos}/{matrix_total}] {desc}"
            pair_pbar = tqdm(total=total_pairs, desc=desc, leave=False)

        for start in range(0, total_pairs, batch_size):
            end = min(start + batch_size, total_pairs)
            batch_pairs = pairs[start:end]
            batch_scores = model.predict(batch_pairs)
            all_scores.extend(batch_scores)

            if pair_pbar is not None:
                pair_pbar.update(end - start)

        if pair_pbar is not None:
            pair_pbar.close()

        for (i, j), score in zip(index_pairs, all_scores):
            mat[i, j] = score
            mat[j, i] = score

    return mat

In [7]:
# ===== 主处理 =====

all_matrices = {}  # 每个题目对应9个矩阵

question_names = list(all_results.keys())
num_questions = len(question_names)
num_matrices = len(MAT_NAMES)
total_matrix_jobs = num_questions * num_matrices

overall_pbar = tqdm(total=total_matrix_jobs, desc="All matrices", position=0)

for q_idx, qname in enumerate(tqdm(question_names, desc="Questions", position=1), start=1):
    qdata = all_results[qname]
    mats = {}

    per_question_pbar = tqdm(
        total=num_matrices,
        desc=f"Question {q_idx}/{num_questions}: {qname}",
        position=2,
        leave=False
    )

    for mat_idx, (key, mat_name) in enumerate(zip(KEY_ORDER, MAT_NAMES), start=1):
        texts = qdata[key]

        per_question_pbar.set_postfix_str(f"{mat_idx}/{num_matrices} -> {mat_name}")
        overall_pbar.set_postfix_str(f"{qname} | {mat_name}")

        mat = build_similarity_matrix(
            texts,
            batch_size=64,
            show_pair_progress=True,
            matrix_label=f"{qname} | {mat_name}",
            matrix_pos=mat_idx,
            matrix_total=num_matrices
        )
        mats[mat_name] = mat

        per_question_pbar.update(1)
        overall_pbar.update(1)

    per_question_pbar.close()
    all_matrices[qname] = mats

overall_pbar.close()

All matrices:   0%|          | 0/162 [00:00<?, ?it/s]

Questions:   0%|          | 0/18 [00:00<?, ?it/s]

Question 1/18: Easy B3666:   0%|          | 0/9 [00:00<?, ?it/s]

[matrix 1/9] Easy B3666 | mat1PI:   0%|          | 0/3160 [00:00<?, ?it/s]

[matrix 2/9] Easy B3666 | mat2FD:   0%|          | 0/3160 [00:00<?, ?it/s]

...

Question 14/18: Mid P1407:   0%|          | 0/9 [00:00<?, ?it/s]

Question 15/18: Mid P14989:   0%|          | 0/9 [00:00<?, ?it/s]

Question 16/18: Mid P3007:   0%|          | 0/9 [00:00<?, ?it/s]

Question 17/18: Mid P3167:   0%|          | 0/9 [00:00<?, ?it/s]

Question 18/18: Mid P4092:   0%|          | 0/9 [00:00<?, ?it/s]

In [8]:
# all_matrices打印辅助函数
def get_question_names_from_all_matrices(all_matrices):
    return list(all_matrices.keys())

def get_question_name_by_index(all_matrices, q_index):
    qnames = get_question_names_from_all_matrices(all_matrices)
    if not (0 <= q_index < len(qnames)):
        raise IndexError(f"q_index={q_index} 越界，当前题目总数为 {len(qnames)}")
    return qnames[q_index]

def print_question_matrices(all_matrices, q_index, selected_mats=None, round_digits=6):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]

    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    print(f"题目 index = {q_index}")
    print(f"题目名称 = {qname}")

    for mat_name in mat_names:
        print(f"\n===== {mat_name} =====")
        mat = mats[mat_name]
        print(f"shape = {mat.shape}")
        display(pd.DataFrame(mat).round(round_digits))

In [10]:
# =============================
# 三种分析方法：辅助函数
# =============================

import numpy as np
import pandas as pd

def check_matrix_and_build_groups(mat, k, temperature_list):
    n = mat.shape[0]
    expected_n = k * len(temperature_list)

    if mat.shape[0] != mat.shape[1]:
        raise ValueError(f"矩阵不是方阵，shape={mat.shape}")

    if n != expected_n:
        raise ValueError(
            f"矩阵大小与参数不匹配：matrix_n={n}, "
            f"但 k * len(temperature_list) = {k} * {len(temperature_list)} = {expected_n}"
        )

    # 顺序约定：
    # [0:k) -> temperature_list[0]
    # [k:2k) -> temperature_list[1]
    # ...
    temp_to_indices = {}
    for ti, temp in enumerate(temperature_list):
        start = ti * k
        end = start + k
        temp_to_indices[temp] = list(range(start, end))

    return temp_to_indices


def get_offdiag_values(mat):
    n = mat.shape[0]
    mask = ~np.eye(n, dtype=bool)
    return mat[mask]


def get_upper_triangle_values(submat):
    n = submat.shape[0]
    vals = []
    for i in range(n):
        for j in range(i + 1, n):
            vals.append(submat[i, j])
    return np.array(vals, dtype=float)


# -----------------------------
# 方法1：整体稳定性
# -----------------------------
def method1_global_stats(mat):
    vals = get_offdiag_values(mat)

    return {
        "offdiag_mean": float(np.mean(vals)),
        "offdiag_std": float(np.std(vals)),
        "offdiag_min": float(np.min(vals)),
        "offdiag_max": float(np.max(vals)),
    }


# -----------------------------
# 方法2：温度内 / 温度间分解
# -----------------------------
def method2_temperature_decomposition(mat, k, temperature_list):
    temp_to_indices = check_matrix_and_build_groups(mat, k, temperature_list)

    # 1) 每个温度内部（within-temp）
    within_rows = []
    within_all_vals = []

    for temp in temperature_list:
        idxs = temp_to_indices[temp]
        submat = mat[np.ix_(idxs, idxs)]
        vals = get_upper_triangle_values(submat)

        within_rows.append({
            "temperature": temp,
            "sample_indices": idxs,
            "within_mean": float(np.mean(vals)) if len(vals) > 0 else np.nan,
            "within_std": float(np.std(vals)) if len(vals) > 0 else np.nan,
            "within_min": float(np.min(vals)) if len(vals) > 0 else np.nan,
            "within_max": float(np.max(vals)) if len(vals) > 0 else np.nan,
        })

        within_all_vals.extend(vals.tolist())

    within_df = pd.DataFrame(within_rows)

    # 2) 不同温度之间（cross-temp）
    cross_mean = pd.DataFrame(index=temperature_list, columns=temperature_list, dtype=float)
    cross_std = pd.DataFrame(index=temperature_list, columns=temperature_list, dtype=float)

    cross_all_vals = []

    for i, temp_i in enumerate(temperature_list):
        idx_i = temp_to_indices[temp_i]

        for j, temp_j in enumerate(temperature_list):
            idx_j = temp_to_indices[temp_j]

            if i == j:
                cross_mean.loc[temp_i, temp_j] = np.nan
                cross_std.loc[temp_i, temp_j] = np.nan
                continue

            vals = []
            for a in idx_i:
                for b in idx_j:
                    vals.append(mat[a, b])

            vals = np.array(vals, dtype=float)
            cross_mean.loc[temp_i, temp_j] = float(np.mean(vals))
            cross_std.loc[temp_i, temp_j] = float(np.std(vals))

            # 只累计上三角温度对，避免重复
            if i < j:
                cross_all_vals.extend(vals.tolist())

    overall_within_mean = float(np.mean(within_all_vals)) if len(within_all_vals) > 0 else np.nan
    overall_cross_mean = float(np.mean(cross_all_vals)) if len(cross_all_vals) > 0 else np.nan

    summary = {
        "overall_within_mean": overall_within_mean,
        "overall_cross_mean": overall_cross_mean,
        "within_minus_cross": (
            overall_within_mean - overall_cross_mean
            if (not np.isnan(overall_within_mean) and not np.isnan(overall_cross_mean))
            else np.nan
        )
    }

    return {
        "within_df": within_df,
        "cross_mean_df": cross_mean,
        "cross_std_df": cross_std,
        "summary": summary,
    }


# -----------------------------
# 方法3：原型 / 离群分析
# -----------------------------
def method3_prototype_outlier(mat, k=None, temperature_list=None):
    n = mat.shape[0]

    row_mean_offdiag = (np.sum(mat, axis=1) - np.diag(mat)) / (n - 1)

    df = pd.DataFrame({
        "sample_index": np.arange(n),
        "row_mean_offdiag": row_mean_offdiag,
    })

    if k is not None and temperature_list is not None:
        expected_n = k * len(temperature_list)
        if expected_n == n:
            df["temperature"] = [temperature_list[i // k] for i in range(n)]
            df["local_index_in_temp"] = [i % k for i in range(n)]

    prototype_idx = int(np.argmax(row_mean_offdiag))
    outlier_idx = int(np.argmin(row_mean_offdiag))

    df = df.sort_values("row_mean_offdiag", ascending=False).reset_index(drop=True)

    return {
        "row_mean_df": df,
        "prototype_idx": prototype_idx,
        "prototype_score": float(row_mean_offdiag[prototype_idx]),
        "outlier_idx": outlier_idx,
        "outlier_score": float(row_mean_offdiag[outlier_idx]),
    }


# -----------------------------
# 汇总：单个矩阵三种分析
# -----------------------------
def analyze_one_matrix(mat, k, temperature_list):
    m1 = method1_global_stats(mat)
    m2 = method2_temperature_decomposition(mat, k, temperature_list)
    m3 = method3_prototype_outlier(mat, k=k, temperature_list=temperature_list)

    overview = {
        **m1,
        **m2["summary"],
        "prototype_idx": m3["prototype_idx"],
        "prototype_score": m3["prototype_score"],
        "outlier_idx": m3["outlier_idx"],
        "outlier_score": m3["outlier_score"],
    }

    return {
        "overview": overview,
        "method1": m1,
        "method2": m2,
        "method3": m3,
    }

In [11]:
# =============================
# 对指定题目的所有矩阵做分析（中文版）
# =============================

def analyze_question_matrices(all_matrices, q_index, k, temperature_list, selected_mats=None, round_digits=6):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]

    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    print(f"题目 index = {q_index}")
    print(f"题目名称 = {qname}")
    print(f"k = {k}")
    print(f"temperature_list = {temperature_list}")

    overview_rows = []
    stat_results = {}

    for mat_name in mat_names:
        mat = mats[mat_name]
        result = analyze_one_matrix(mat, k, temperature_list)
        stat_results[mat_name] = result

        overview_row = {"matrix": mat_name}
        overview_row.update(result["overview"])
        overview_rows.append(overview_row)

    overview_df = pd.DataFrame(overview_rows)
    print("\n===== 总览表 =====")
    display(overview_df.round(round_digits))

    for mat_name in mat_names:
        result = stat_results[mat_name]

        print(f"\n\n==============================")
        print(f"矩阵: {mat_name}")
        print(f"==============================")

        print("\n[方法1] 整体稳定性（非对角元素统计）")
        display(pd.DataFrame([result["method1"]]).round(round_digits))

        print("\n[方法2-A] 温度内稳定性")
        display(result["method2"]["within_df"].round(round_digits))

        print("\n[方法2-B] 温度间均值矩阵")
        display(result["method2"]["cross_mean_df"].round(round_digits))

        print("\n[方法2-C] 温度间标准差矩阵")
        display(result["method2"]["cross_std_df"].round(round_digits))

        print("\n[方法2-D] 温度分解汇总")
        display(pd.DataFrame([result["method2"]["summary"]]).round(round_digits))

        print("\n[方法3] 原型 / 离群分析（按 row mean offdiag 排序）")
        display(result["method3"]["row_mean_df"].round(round_digits))

        print(
            f"prototype_idx = {result['method3']['prototype_idx']}, "
            f"prototype_score = {result['method3']['prototype_score']:.{round_digits}f}"
        )
        print(
            f"outlier_idx = {result['method3']['outlier_idx']}, "
            f"outlier_score = {result['method3']['outlier_score']:.{round_digits}f}"
        )

    return overview_df, stat_results

题目 index = 0
题目名称 = Easy B3666
k = 20
temperature_list = [0.2, 0.5, 0.8, 1.0]

===== 总览表 =====


,matrix,offdiag_mean,offdiag_std,offdiag_min,offdiag_max,overall_within_mean,overall_cross_mean,within_minus_cross,prototype_idx,prototype_score,outlier_idx,outlier_score
0,mat1PI,0.757690,0.058490,0.600664,0.932416,0.793663,0.746298,0.047366,0,0.790566,73,0.678953
1,mat2FD,0.737014,0.080047,0.513030,0.937798,0.759622,0.729855,0.029767,0,0.803925,39,0.625806
2,mat3RC,0.635470,0.105595,0.399533,0.962741,0.701351,0.614607,0.086744,1,0.701264,74,0.493384
3,mat4MS,0.708719,0.068221,0.537351,0.909813,0.735524,0.700230,0.035294,0,0.765795,64,0.623357
4,mat5INV,0.692119,0.088029,0.496809,0.955818,0.729639,0.680237,0.049402,0,0.770258,65,0.587802
5,mat6POUR,0.661442,0.068497,0.527780,0.895031,0.714744,0.644563,0.070181,0,0.699297,55,0.602677
6,mat7OR,0.708271,0.083142,0.524267,0.969586,0.766809,0.689734,0.077075,63,0.762219,48,0.631991
7,mat8ECCC,0.575143,0.064746,0.436574,0.800816,0.613537,0.562986,0.050551,51,0.663793,76,0.514582
8,mat9N,0.543311,0.245823,0.009193,0.972853,0.618194,0.519598,0.098595,50,0.648622,63,0.254095




矩阵: mat1PI

[方法1] 整体稳定性（非对角元素统计）


,offdiag_mean,offdiag_std,offdiag_min,offdiag_max
0,0.75769,0.05849,0.600664,0.932416



[方法2-A] 温度内稳定性


...


[方法2-D] 温度分解汇总


,overall_within_mean,overall_cross_mean,within_minus_cross
0,0.618194,0.519598,0.098595



[方法3] 原型 / 离群分析（按 row mean offdiag 排序）


,sample_index,row_mean_offdiag,temperature,local_index_in_temp
0,50,0.648622,0.8,10
1,0,0.640962,0.2,0
2,2,0.640962,0.2,2
3,1,0.640962,0.2,1
4,4,0.640962,0.2,4
...,...,...,...,...
75,69,0.335772,1.0,9
76,74,0.311847,1.0,14
77,31,0.291657,0.5,11
78,70,0.285909,1.0,10


prototype_idx = 50, prototype_score = 0.648622
outlier_idx = 63, outlier_score = 0.254095


In [12]:
# =============================
# 对指定题目的所有矩阵做分析（英文版）
# =============================

def analyze_question_matrices(all_matrices, q_index, k, temperature_list, selected_mats=None, round_digits=6):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]

    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    print(f"Question index = {q_index}")
    print(f"Question Name = {qname}")
    print(f"k = {k}")
    print(f"temperature_list = {temperature_list}")

    overview_rows = []
    stat_results = {}

    for mat_name in mat_names:
        mat = mats[mat_name]
        result = analyze_one_matrix(mat, k, temperature_list)
        stat_results[mat_name] = result

        overview_row = {"matrix": mat_name}
        overview_row.update(result["overview"])
        overview_rows.append(overview_row)

    overview_df = pd.DataFrame(overview_rows)
    print("\n===== Overview Table =====")
    display(overview_df.round(round_digits))

    for mat_name in mat_names:
        result = stat_results[mat_name]

        print(f"\n\n==============================")
        print(f"Matrix: {mat_name}")
        print(f"==============================")

        print("\n[Method 1] Overall Stability (Off-Diagonal Statistics)")
        display(pd.DataFrame([result["method1"]]).round(round_digits))

        print("\n[Method 2-A] Within-Temperature Stability")
        display(result["method2"]["within_df"].round(round_digits))

        print("\n[Method 2-B] Between-Temperature Mean Matrix")
        display(result["method2"]["cross_mean_df"].round(round_digits))

        print("\n[Method 2-C] Between-Temperature Standard Deviation Matrix")
        display(result["method2"]["cross_std_df"].round(round_digits))

        print("\n[Method 2-D] Summary of Temperature Decomposition")
        display(pd.DataFrame([result["method2"]["summary"]]).round(round_digits))

        print("\n[Method 3] Prototype / Outlier Analysis (sorted by row mean offdiag)")
        display(result["method3"]["row_mean_df"].round(round_digits))

        print(
            f"prototype_idx = {result['method3']['prototype_idx']}, "
            f"prototype_score = {result['method3']['prototype_score']:.{round_digits}f}"
        )
        print(
            f"outlier_idx = {result['method3']['outlier_idx']}, "
            f"outlier_score = {result['method3']['outlier_score']:.{round_digits}f}"
        )

    return overview_df, stat_results

In [13]:
# =========================================================
# 路径辅助
# =========================================================
def get_question_dir_by_name(question_name, questions_dir=QUESTIONS_DIR):
    qdir = Path(questions_dir) / question_name
    qdir.mkdir(parents=True, exist_ok=True)
    return qdir


def save_df_csv(df, path, round_digits=6):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=True, encoding="utf-8-sig", float_format=f"%.{round_digits}f")


def save_dict_as_one_row_csv(d, path, round_digits=6):
    df = pd.DataFrame([d])
    save_df_csv(df, path, round_digits=round_digits)


# =========================================================
# 1) 单题：打印矩阵 + 保存矩阵
# 保存位置：
#   Questions/<题目名>/CrossEncoder_Results/matrices/
# =========================================================
def print_and_save_question_matrices(
    all_matrices,
    q_index,
    questions_dir=QUESTIONS_DIR,
    selected_mats=None,
    round_digits=6,
    save_npy=True,
    save_csv=True,
):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]
    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    qdir = get_question_dir_by_name(qname, questions_dir)
    out_dir = qdir / "CrossEncoder_Results" / "matrices"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*80}")
    print(f"题目 index = {q_index}")
    print(f"题目名称 = {qname}")
    print(f"矩阵保存目录 = {out_dir}")
    print(f"{'='*80}")

    for mat_name in mat_names:
        mat = mats[mat_name]
        df = pd.DataFrame(mat)

        print(f"\n===== {mat_name} =====")
        print(f"shape = {mat.shape}")
        display(df.round(round_digits))

        if save_npy:
            np.save(out_dir / f"{mat_name}.npy", mat)

        if save_csv:
            df.to_csv(
                out_dir / f"{mat_name}.csv",
                index=True,
                encoding="utf-8-sig",
                float_format=f"%.{round_digits}f"
            )


# =========================================================
# 2) 单题：分析矩阵 + 打印分析 + 保存分析
# 保存位置：
#   Questions/<题目名>/CrossEncoder_Results/analysis/
# =========================================================
def analyze_and_save_question_matrices(
    all_matrices,
    q_index,
    k,
    temperature_list,
    questions_dir=QUESTIONS_DIR,
    selected_mats=None,
    round_digits=6,
):
    qname = get_question_name_by_index(all_matrices, q_index)
    mats = all_matrices[qname]
    mat_names = selected_mats if selected_mats is not None else list(mats.keys())

    qdir = get_question_dir_by_name(qname, questions_dir)
    out_dir = qdir / "CrossEncoder_Results" / "analysis"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*80}")
    print(f"Question index = {q_index}")
    print(f"Question Name = {qname}")
    print(f"k = {k}")
    print(f"temperature_list = {temperature_list}")
    print(f"analysis save dir = {out_dir}")
    print(f"{'='*80}")

    overview_rows = []
    stat_results = {}

    for mat_name in mat_names:
        mat = mats[mat_name]
        result = analyze_one_matrix(mat, k, temperature_list)
        stat_results[mat_name] = result

        overview_row = {"matrix": mat_name}
        overview_row.update(result["overview"])
        overview_rows.append(overview_row)

    overview_df = pd.DataFrame(overview_rows)

    print("\n===== Overview Table =====")
    display(overview_df.round(round_digits))

    # 保存 overview
    save_df_csv(overview_df, out_dir / "overview.csv", round_digits=round_digits)

    # 每个矩阵单独保存
    for mat_name in mat_names:
        result = stat_results[mat_name]
        mat_dir = out_dir / mat_name
        mat_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n\n==============================")
        print(f"Matrix: {mat_name}")
        print(f"==============================")

        print("\n[Method 1] Overall Stability (Off-Diagonal Statistics)")
        method1_df = pd.DataFrame([result["method1"]])
        display(method1_df.round(round_digits))

        print("\n[Method 2-A] Within-Temperature Stability")
        within_df = result["method2"]["within_df"]
        display(within_df.round(round_digits))

        print("\n[Method 2-B] Between-Temperature Mean Matrix")
        cross_mean_df = result["method2"]["cross_mean_df"]
        display(cross_mean_df.round(round_digits))

        print("\n[Method 2-C] Between-Temperature Standard Deviation Matrix")
        cross_std_df = result["method2"]["cross_std_df"]
        display(cross_std_df.round(round_digits))

        print("\n[Method 2-D] Summary of Temperature Decomposition")
        method2_summary_df = pd.DataFrame([result["method2"]["summary"]])
        display(method2_summary_df.round(round_digits))

        print("\n[Method 3] Prototype / Outlier Analysis (sorted by row mean offdiag)")
        row_mean_df = result["method3"]["row_mean_df"]
        display(row_mean_df.round(round_digits))

        print(
            f"prototype_idx = {result['method3']['prototype_idx']}, "
            f"prototype_score = {result['method3']['prototype_score']:.{round_digits}f}"
        )
        print(
            f"outlier_idx = {result['method3']['outlier_idx']}, "
            f"outlier_score = {result['method3']['outlier_score']:.{round_digits}f}"
        )

        # 保存各表
        save_df_csv(method1_df, mat_dir / "method1_global_stats.csv", round_digits=round_digits)
        save_df_csv(within_df, mat_dir / "method2_within_temperature.csv", round_digits=round_digits)
        save_df_csv(cross_mean_df, mat_dir / "method2_cross_mean.csv", round_digits=round_digits)
        save_df_csv(cross_std_df, mat_dir / "method2_cross_std.csv", round_digits=round_digits)
        save_df_csv(method2_summary_df, mat_dir / "method2_summary.csv", round_digits=round_digits)
        save_df_csv(row_mean_df, mat_dir / "method3_row_mean_offdiag.csv", round_digits=round_digits)

        # 额外保存一个简短文本摘要
        summary_txt = []
        summary_txt.append(f"Question Name: {qname}")
        summary_txt.append(f"Matrix: {mat_name}")
        summary_txt.append("")
        summary_txt.append("[Method 1] Overall Stability")
        for k1, v1 in result["method1"].items():
            summary_txt.append(f"{k1}: {v1:.{round_digits}f}")
        summary_txt.append("")
        summary_txt.append("[Method 2] Temperature Decomposition Summary")
        for k2, v2 in result["method2"]["summary"].items():
            if isinstance(v2, (int, float, np.floating)):
                summary_txt.append(f"{k2}: {v2:.{round_digits}f}")
            else:
                summary_txt.append(f"{k2}: {v2}")
        summary_txt.append("")
        summary_txt.append("[Method 3] Prototype / Outlier")
        summary_txt.append(
            f"prototype_idx: {result['method3']['prototype_idx']}, "
            f"prototype_score: {result['method3']['prototype_score']:.{round_digits}f}"
        )
        summary_txt.append(
            f"outlier_idx: {result['method3']['outlier_idx']}, "
            f"outlier_score: {result['method3']['outlier_score']:.{round_digits}f}"
        )

        (mat_dir / "summary.txt").write_text("\n".join(summary_txt), encoding="utf-8")

    return overview_df, stat_results


# =========================================================
# 3) 所有题目：批量打印 + 保存
# =========================================================
def run_all_questions_print_and_save(
    all_matrices,
    k,
    temperature_list,
    questions_dir=QUESTIONS_DIR,
    selected_mats=None,
    round_digits=6,
    save_matrices=True,
    save_analysis=True,
):
    all_question_overviews = []

    qnames = get_question_names_from_all_matrices(all_matrices)

    for q_index, qname in enumerate(qnames):
        print(f"\n\n{'#'*100}")
        print(f"Processing question {q_index}/{len(qnames)-1}: {qname}")
        print(f"{'#'*100}")

        if save_matrices:
            print_and_save_question_matrices(
                all_matrices=all_matrices,
                q_index=q_index,
                questions_dir=questions_dir,
                selected_mats=selected_mats,
                round_digits=round_digits,
                save_npy=True,
                save_csv=True,
            )

        if save_analysis:
            overview_df, stat_results = analyze_and_save_question_matrices(
                all_matrices=all_matrices,
                q_index=q_index,
                k=k,
                temperature_list=temperature_list,
                questions_dir=questions_dir,
                selected_mats=selected_mats,
                round_digits=round_digits,
            )

            tmp = overview_df.copy()
            tmp.insert(0, "question_name", qname)
            tmp.insert(0, "question_index", q_index)
            all_question_overviews.append(tmp)

    # 额外保存一个总汇总表（放在 Questions 根目录下）
    if all_question_overviews:
        global_overview_df = pd.concat(all_question_overviews, ignore_index=True)
        global_path = Path(questions_dir) / "CrossEncoder_Results_AllQuestions_Overview.csv"
        global_overview_df.to_csv(
            global_path,
            index=False,
            encoding="utf-8-sig",
            float_format=f"%.{round_digits}f"
        )
        print(f"\nAll-question overview saved to: {global_path}")

        return global_overview_df

    return None

In [14]:
global_overview_df = run_all_questions_print_and_save(
    all_matrices=all_matrices,
    k=k,
    temperature_list=temperature_list,
    questions_dir=QUESTIONS_DIR,
    selected_mats=SELECTED_MATS,
    round_digits=ROUND_DIGITS,
    save_matrices=True,
    save_analysis=True,
)



####################################################################################################
Processing question 0/17: Easy B3666
####################################################################################################

题目 index = 0
题目名称 = Easy B3666
矩阵保存目录 = Questions/Easy B3666/CrossEncoder_Results/matrices

===== mat1PI =====
shape = (80, 80)


,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
0,1.000000,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,...,0.742610,0.720573,0.727237,0.743048,0.737616,0.782865,0.753669,0.725956,0.772721,0.773219
1,0.932416,1.000000,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,...,0.742610,0.720573,0.727237,0.743048,0.737616,0.782865,0.753669,0.725956,0.772721,0.773219
2,0.932416,0.932416,1.000000,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,...,0.742610,0.720573,0.727237,0.743048,0.737616,0.782865,0.753669,0.725956,0.772721,0.773219
3,0.932416,0.932416,0.932416,1.000000,0.932416,0.932416,0.932416,0.932416,0.932416,0.932416,...,0.742610,0.720573,0.727237,0.743048,0.737616,0.782865,0.753669,0.725956,0.772721,0.773219
4,0.932416,0.932416,0.932416,0.932416,1.000000,0.932416,0.932416,0.932416,0.932416,0.932416,...,0.742610,0.720573,0.727237,0.743048,0.737616,0.782865,0.753669,0.725956,0.772721,0.773219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.782865,0.782865,0.782865,0.782865,0.782865,0.782865,0.782865,0.782865,0.782865,0.782865,...,0.713352,0.725319,0.732733,0.785111,0.782900,1.000000,0.747704,0.752717,0.812483,0.786229
76,0.753669,0.753669,0.753669,0.753669,0.753669,0.753669,0.753669,0.753669,0.753669,0.753669,...,0.759771,0.740572,0.705982,0.740654,0.828167,0.747704,1.000000,0.832080,0.801342,0.762384
77,0.725956,0.725956,0.725956,0.725956,0.725956,0.725956,0.725956,0.725956,0.725956,0.725956,...,0.699156,0.749979,0.722023,0.750549,0.797293,0.752717,0.832080,1.000000,0.759443,0.765376
78,0.772721,0.772721,0.772721,0.772721,0.772721,0.772721,0.772721,0.772721,0.772721,0.772721,...,0.740559,0.762598,0.696816,0.785378,0.818542,0.812483,0.801342,0.759443,1.000000,0.712056



===== mat2FD =====
shape = (80, 80)


,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
0,1.000000,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,...,0.818189,0.667240,0.812937,0.745534,0.750655,0.786652,0.806821,0.752666,0.800093,0.822185
1,0.937798,1.000000,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,...,0.818189,0.667240,0.812937,0.745534,0.750655,0.786652,0.806821,0.752666,0.800093,0.822185
2,0.937798,0.937798,1.000000,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,...,0.818189,0.667240,0.812937,0.745534,0.750655,0.786652,0.806821,0.752666,0.800093,0.822185
3,0.937798,0.937798,0.937798,1.000000,0.937798,0.937798,0.937798,0.937798,0.937798,0.937798,...,0.818189,0.667240,0.812937,0.745534,0.750655,0.786652,0.806821,0.752666,0.800093,0.822185
4,0.937798,0.937798,0.937798,0.937798,1.000000,0.937798,0.937798,0.937798,0.937798,0.937798,...,0.818189,0.667240,0.812937,0.745534,0.750655,0.786652,0.806821,0.752666,0.800093,0.822185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0.786652,0.786652,0.786652,0.786652,0.786652,0.786652,0.786652,0.786652,0.786652,0.786652,...,0.726473,0.572207,0.768425,0.677656,0.681837,1.000000,0.630828,0.656737,0.738551,0.682232
76,0.806821,0.806821,0.806821,0.806821,0.806821,0.806821,0.806821,0.806821,0.806821,0.806821,...,0.680583,0.542165,0.741109,0.602614,0.712777,0.630828,1.000000,0.759166,0.757097,0.768961
77,0.752666,0.752666,0.752666,0.752666,0.752666,0.752666,0.752666,0.752666,0.752666,0.752666,...,0.673025,0.600096,0.728700,0.603259,0.716913,0.656737,0.759166,1.000000,0.668635,0.669421
78,0.800093,0.800093,0.800093,0.800093,0.800093,0.800093,0.800093,0.800093,0.800093,0.800093,...,0.758054,0.574225,0.780849,0.618966,0.720292,0.738551,0.757097,0.668635,1.000000,0.745520



===== mat3RC =====
shape = (80, 80)


...


===== mat9N =====
shape = (0, 0)


""



Question index = 5
Question Name = Hard P11658
k = 20
temperature_list = [0.0, 0.2, 0.5, 0.8]
analysis save dir = Questions/Hard P11658/CrossEncoder_Results/analysis


/opt/conda/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/conda/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/conda/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/conda/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/conda/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: zero-size array to reduction operation minimum which has no identity